<a href="https://colab.research.google.com/github/avundukarda/AvundukArda-Programming-BigDataAnalytics/blob/main/3_02_key_value_stores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1vv_PsWBnUJwSCkwKDoJAC-vXjtaEA4Ts)

# 3.02 Key-value Stores with TinyDB
This tutorial gives a basic introduction to working with key-value (KV) stores (or document DBs). We will be working with [TinyDB](https://tinydb.readthedocs.io/en/latest/index.html), an in-memory Python database, which is particularly attractive here as it is, as the name suggests, pretty small and lightweight.

We will begin with the relevant installs:

In [9]:
!pip install tinydb
!pip install faker
!pip install python-lorem

As you may infer from the pacakages installed, we will run something similar to one of our DuckDB examples - specifically building a database using fake data generated by Faker:

In [10]:
import random
from faker import Faker
import pandas as pd
from lorem import paragraph
import itertools

fake = Faker()

def get_person():
  person = {}
  person['id'] = random.randrange(1000,9999999999999)
  person['first_name'] = fake.first_name()
  person['last_name'] = fake.last_name()
  person['email'] = fake.unique.ascii_email()
  person['company'] = fake.company()
  person['phone'] = fake.phone_number()
  person['review'] = list(itertools.islice(paragraph(count=1), 1))
  return person

personlist = []
for x in range(100):
  personlist.append(get_person())

df = pd.DataFrame.from_dict(personlist)
df.head()

,id,first_name,last_name,email,company,phone,review
0,500816676745,Rebecca,Mccormick,leblanchannah@yahoo.com,Dixon LLC,377-731-7396,"[Culpa non laboris magna excepteur, enim ullam..."
1,8566297885265,Maria,Williams,rossrobert@yahoo.com,Wall LLC,+1-490-541-3341x42049,[Excepteur aliqua dolor amet. Deserunt duis al...
2,1758018660248,Kristen,Martinez,adrianrichard@hotmail.com,"Watson, Jackson and Moreno",399-841-3550,[Consectetur id laboris aliqua non aliqua culp...
3,4924815970401,Craig,Rodriguez,brian77@yahoo.com,"Jones, Smith and Pierce",001-655-558-6683,[Exercitation id incididunt sint esse occaecat...
4,3312516517318,David,Whitney,eruiz@peck.com,Stephens-Dawson,943-230-3638x98333,"[Voluptate mollit excepteur sit aute, laborum ..."


Everything here is the same except we have also add a text column (using lorem ipsum). As before we have created this as a Pandas dataframe, but like most KV stores, TinyDB prefers data stored as a dictionary:

In [11]:
fake_data = df.to_dict(orient='records')
fake_data

[{'id': 500816676745,
  'first_name': 'Rebecca',
  'last_name': 'Mccormick',
  'email': 'leblanchannah@yahoo.com',
  'company': 'Dixon LLC',
  'phone': '377-731-7396',
  'review': ['Culpa non laboris magna excepteur, enim ullamco velit in anim amet. Sint fugiat lorem pariatur irure sit ad. Mollit proident non incididunt laboris consequat. Officia adipiscing proident ut et voluptate ullamco. Excepteur enim ipsum magna.']},
 {'id': 8566297885265,
  'first_name': 'Maria',
  'last_name': 'Williams',
  'email': 'rossrobert@yahoo.com',
  'company': 'Wall LLC',
  'phone': '+1-490-541-3341x42049',
  'review': ['Excepteur aliqua dolor amet. Deserunt duis aliquip culpa ad ex, cupidatat officia pariatur dolore enim quis. Officia mollit ad laborum nisi dolore do consequat. Consectetur incididunt pariatur proident voluptate laborum, laboris est in in. Amet sunt nulla amet irure magna. Voluptate tempor pariatur exercitation id qui. Incididunt nostrud elit est.']},
 {'id': 1758018660248,
  'first_nam

With this transform in place we can load the data into our database. You may note the database itself is specified as JSON format:

In [12]:
from tinydb import TinyDB, Query

db = TinyDB('db.json')

for record in fake_data:
  db.insert(record)

We can check this has worked with a simple Python loop:

In [13]:
for item in db:
  print(item)

{'id': 1959104168842, 'first_name': 'Jason', 'last_name': 'Smith', 'email': 'kdennis@nelson.net', 'company': 'Rogers-Todd', 'phone': '957-272-8826x64413', 'review': ['Minim magna duis qui tempor ad. Occaecat ullamco quis ullamco proident elit, laboris exercitation duis ex ullamco ullamco et. Ullamco nostrud eiusmod sint veniam. Ullamco adipiscing sunt sit adipiscing. Voluptate officia dolor et sit nulla cupidatat. Reprehenderit qui voluptate velit qui.']}
{'id': 912691768198, 'first_name': 'Jay', 'last_name': 'Johnson', 'email': 'jeanette55@yahoo.com', 'company': 'White-Shields', 'phone': '9635819839', 'review': ['Consequat deserunt fugiat fugiat magna laboris veniam, eiusmod non mollit id irure. Id aliqua ut et officia, exercitation incididunt ullamco lorem. Excepteur reprehenderit dolore adipiscing enim amet culpa. Cillum anim reprehenderit voluptate. Ex fugiat quis pariatur. Aute incididunt culpa laborum velit in. Est pariatur fugiat occaecat commodo sit in, labore non velit qui con

With our database setup, we can start to query our records. In TinyDB we do this by creating a query object:

In [16]:
User = Query() # query object

db.search(User.first_name == 'Chad') # adapt based on your data

[{'id': 1131084269801,
  'first_name': 'Chad',
  'last_name': 'Martinez',
  'email': 'gallen@rodriguez.biz',
  'company': 'Kelly-Taylor',
  'phone': '+1-648-536-5846',
  'review': ['Cupidatat voluptate sed dolore ea pariatur proident. Lorem ad tempor ullamco labore, ullamco lorem lorem irure aliqua consectetur. Quis laboris id qui officia lorem. Veniam ex nisi consectetur laborum. Fugiat duis elit eu anim, elit incididunt sunt exercitation. Consectetur excepteur labore aliqua, fugiat eiusmod deserunt dolore sunt anim enim, est pariatur est et voluptate incididunt. Adipiscing esse nulla commodo magna magna nisi deserunt. Voluptate sint sit excepteur nulla laborum, aute veniam consectetur minim sed sed amet minim, culpa sint eiusmod in. Pariatur eiusmod ullamco ipsum in minim irure.']}]

We can also add new data in dictionary/JSON-like format:

In [17]:
db.insert({'id': 123, 'first_name': 'Amir', 'star_sign': 'Dog', 'review': 'I do not speak Latin.'})

201

And retrieve the data as before:

In [18]:
db.search(User.id == 123)

[{'id': 123,
  'first_name': 'Amir',
  'star_sign': 'Dog',
  'review': 'I do not speak Latin.'}]

One thing to note here is that our new record does not follow the schema we may infer from the original dataset (i.e. the original data all used the same columns/fields). Here many of those fields are missing and we have the new field 'star_sign'.

This demonstrates the extra flexibility we get with a KV store over a relational model. We can also query our database to get all records that have a specific field:

In [19]:
db.search(User.star_sign.exists())

[{'id': 123,
  'first_name': 'Amir',
  'star_sign': 'Dog',
  'review': 'I do not speak Latin.'}]

This gives a basic intro into KV (and document) stores. While there are many competing brands/solutions, the common themes are the dictionary-like structure (key-value pairs) and flexibility to accept any fields (keys).